# Laboratório 07 — Especialização de LLMs com LoRA e QLoRA

Pipeline completo de fine-tuning do modelo **Llama 2 7B** com quantização 4-bit (QLoRA) e adaptadores LoRA, especializado no domínio de **futebol**.

**Autor:** José Lucas Silva Souza

> **Antes de começar:** vá em `Runtime > Change runtime type` e selecione **GPU** (T4 ou superior).

## 0. Verificar GPU disponível

In [ ]:
!nvidia-smi

## 1. Instalar dependências

In [ ]:
!pip install -q \
    openai \
    python-dotenv \
    transformers \
    peft \
    trl \
    bitsandbytes \
    accelerate \
    datasets

## 2. Configurar credenciais

Preencha suas chaves abaixo. Elas ficam apenas na sessão do Colab e não são salvas no notebook.

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "sk-..."        # sua chave OpenAI
os.environ["HF_TOKEN"]       = "hf_..."        # seu token Hugging Face
os.environ["BASE_MODEL"]     = "NousResearch/Llama-2-7b-hf"

## 3. Configurações centralizadas (config.py)

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path

DATA_DIR   = Path("data")
OUTPUT_DIR = Path("outputs/football-lora")


@dataclass
class DatasetSettings:
    domain: str       = "football"
    total_pairs: int  = 60
    train_ratio: float = 0.9
    random_seed: int  = 42
    train_path: Path  = field(default_factory=lambda: DATA_DIR / "train.jsonl")
    test_path: Path   = field(default_factory=lambda: DATA_DIR / "test.jsonl")


@dataclass
class QuantHyperparams:
    load_in_4bit: bool            = True
    bnb_4bit_quant_type: str      = "nf4"
    bnb_4bit_compute_dtype: str   = "float16"
    bnb_4bit_use_double_quant: bool = False


@dataclass
class LoraHyperparams:
    r: int              = 64
    lora_alpha: int     = 16
    lora_dropout: float = 0.1
    bias: str           = "none"
    task_type: str      = "CAUSAL_LM"
    target_modules: list = field(default_factory=lambda: ["q_proj", "v_proj"])


@dataclass
class TrainHyperparams:
    output_dir: Path                  = field(default_factory=lambda: OUTPUT_DIR)
    num_train_epochs: int             = 3
    per_device_train_batch_size: int  = 4
    gradient_accumulation_steps: int  = 1
    optim: str                        = "paged_adamw_32bit"
    learning_rate: float              = 2e-4
    lr_scheduler_type: str            = "cosine"
    warmup_ratio: float               = 0.03
    logging_steps: int                = 25
    fp16: bool                        = True
    max_seq_length: int               = 512
    save_strategy: str                = "no"

print("Configurações carregadas.")

## 4. Geração do dataset sintético (Passo 1)

Gera 60 pares de pergunta/resposta sobre futebol via OpenAI API e salva em `.jsonl`.

> Se quiser pular esta etapa, vá direto para a célula **"Carregar dataset"** — os dados já estão embutidos abaixo.

In [ ]:
import json
import random
from openai import OpenAI

FOOTBALL_CATEGORIES = [
    "rules and regulations of football",
    "history and legendary players of football",
    "tactics and formations in football",
    "major competitions and tournaments in football",
    "records and statistics in football",
    "football culture and fan traditions",
]


def build_generation_prompt(category: str, n: int) -> str:
    return (
        f"Generate {n} question-and-answer pairs about {category}. "
        "Return ONLY a valid JSON array where each element has exactly two keys: "
        '"prompt" (the question or instruction) and "response" (the answer, 2 to 4 sentences, '
        "factually accurate and educational). Do not include any text outside the JSON array."
    )


def generate_pairs_for_category(client, category: str, n: int) -> list:
    message = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": build_generation_prompt(category, n)}],
        temperature=0.7,
    )
    return json.loads(message.choices[0].message.content.strip())


def generate_all_pairs(client, cfg: DatasetSettings) -> list:
    per_category = cfg.total_pairs // len(FOOTBALL_CATEGORIES)
    all_pairs = []
    for category in FOOTBALL_CATEGORIES:
        print(f"  Gerando: {category}...")
        all_pairs.extend(generate_pairs_for_category(client, category, per_category))
    return all_pairs


def split_and_save(pairs: list, cfg: DatasetSettings):
    random.seed(cfg.random_seed)
    shuffled = pairs[:]
    random.shuffle(shuffled)
    cut = int(len(shuffled) * cfg.train_ratio)
    train_pairs, test_pairs = shuffled[:cut], shuffled[cut:]

    for path, data in [(cfg.train_path, train_pairs), (cfg.test_path, test_pairs)]:
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            for pair in data:
                f.write(json.dumps(pair, ensure_ascii=False) + "\n")

    print(f"Train: {len(train_pairs)} pares -> {cfg.train_path}")
    print(f"Test:  {len(test_pairs)} pares  -> {cfg.test_path}")


data_cfg = DatasetSettings()
client   = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print(f"Gerando {data_cfg.total_pairs} pares sobre futebol...")
all_pairs = generate_all_pairs(client, data_cfg)
split_and_save(all_pairs, data_cfg)
print("Dataset gerado com sucesso!")

### Alternativa: carregar dataset do repositório

Se não quiser usar a API da OpenAI, clone o repositório e use os dados prontos:

In [ ]:
# Execute esta célula SOMENTE se quiser usar o dataset do repositório
# (pule se já rodou a geração acima)

!git clone https://github.com/lucasrbsouza/llm-lora-qlora-synthetic-finetuning-lab.git repo_temp

import shutil
shutil.copytree("repo_temp/data", "data", dirs_exist_ok=True)
shutil.rmtree("repo_temp")
print("Dataset copiado de data/")

## 5. Passo 2 — Configuração da Quantização (QLoRA)

Carrega o modelo base em **4-bit NF4** via `bitsandbytes`, reduzindo o consumo de VRAM de ~28 GB para ~6 GB.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training


def build_bnb_config(cfg: QuantHyperparams) -> BitsAndBytesConfig:
    return BitsAndBytesConfig(
        load_in_4bit=cfg.load_in_4bit,
        bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=cfg.bnb_4bit_use_double_quant,
    )


def load_model_and_tokenizer(model_name: str, bnb_config: BitsAndBytesConfig, hf_token: str):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        token=hf_token,
    )
    model = prepare_model_for_kbit_training(model)

    tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
    tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


quant_cfg  = QuantHyperparams()
bnb_config = build_bnb_config(quant_cfg)

BASE_MODEL = os.environ["BASE_MODEL"]
HF_TOKEN   = os.environ.get("HF_TOKEN")

print(f"Carregando modelo: {BASE_MODEL}")
model, tokenizer = load_model_and_tokenizer(BASE_MODEL, bnb_config, HF_TOKEN)
print("Modelo carregado com quantização 4-bit NF4.")

## 6. Passo 3 — Arquitetura LoRA

Aplica adaptadores LoRA com os hiperparâmetros obrigatórios: `r=64`, `lora_alpha=16`, `lora_dropout=0.1`.

In [ ]:
from peft import LoraConfig, get_peft_model


def build_lora_config(cfg: LoraHyperparams) -> LoraConfig:
    return LoraConfig(
        r=cfg.r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        bias=cfg.bias,
        task_type=cfg.task_type,
        target_modules=cfg.target_modules,
    )


lora_cfg    = LoraHyperparams()
lora_config = build_lora_config(lora_cfg)
model       = get_peft_model(model, lora_config)

model.print_trainable_parameters()

## 7. Passo 4 — Pipeline de Treinamento (SFTTrainer)

Usa `SFTTrainer` com otimizador `paged_adamw_32bit`, scheduler `cosine` e `warmup_ratio=0.03`.

In [ ]:
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer


def load_datasets(cfg: DatasetSettings):
    dataset = load_dataset("json", data_files={
        "train": str(cfg.train_path),
        "test":  str(cfg.test_path),
    })
    return dataset["train"], dataset["test"]


def format_prompt(example: dict) -> dict:
    text = f"### Instruction:\n{example['prompt']}\n\n### Response:\n{example['response']}"
    return {"text": text}


def build_training_args(cfg: TrainHyperparams) -> TrainingArguments:
    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    return TrainingArguments(
        output_dir=str(cfg.output_dir),
        num_train_epochs=cfg.num_train_epochs,
        per_device_train_batch_size=cfg.per_device_train_batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        optim=cfg.optim,
        learning_rate=cfg.learning_rate,
        lr_scheduler_type=cfg.lr_scheduler_type,
        warmup_ratio=cfg.warmup_ratio,
        logging_steps=cfg.logging_steps,
        fp16=cfg.fp16,
        save_strategy=cfg.save_strategy,
        report_to="none",
    )


train_cfg = TrainHyperparams()

train_ds, eval_ds = load_datasets(data_cfg)
train_ds = train_ds.map(format_prompt)
eval_ds  = eval_ds.map(format_prompt)

training_args = build_training_args(train_cfg)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=train_cfg.max_seq_length,
)

print("Iniciando treinamento...")
trainer.train()

## 8. Salvar o adaptador LoRA

In [ ]:
trainer.model.save_pretrained(str(train_cfg.output_dir))
tokenizer.save_pretrained(str(train_cfg.output_dir))
print(f"Adaptador salvo em: {train_cfg.output_dir}")

## 9. (Opcional) Fazer download do adaptador

Compacta e baixa o adaptador treinado para sua máquina.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("football-lora-adapter", "zip", str(train_cfg.output_dir))
files.download("football-lora-adapter.zip")

---

## Hiperparâmetros utilizados

| Componente | Parâmetro | Valor |
|------------|-----------|-------|
| Quantização | `bnb_4bit_quant_type` | `nf4` |
| Quantização | `bnb_4bit_compute_dtype` | `float16` |
| Quantização | `load_in_4bit` | `True` |
| LoRA | `r` (rank) | `64` |
| LoRA | `lora_alpha` | `16` |
| LoRA | `lora_dropout` | `0.1` |
| LoRA | `task_type` | `CAUSAL_LM` |
| Treinamento | `optim` | `paged_adamw_32bit` |
| Treinamento | `lr_scheduler_type` | `cosine` |
| Treinamento | `warmup_ratio` | `0.03` |

---

> Partes geradas/complementadas com IA, revisadas por **José Lucas Silva Souza**.